# Imports

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

if Path.cwd().name in ['ETL', 'data_science']:
    root = Path.cwd().parent
else:
    root = Path.cwd()

if str(root) not in sys.path:
    sys.path.append(str(root))
else:
    None

from utils import align_working_dir, init_sys_path,  apply_base_style, CUSTOM_PALETTE, determine_primary_category

align_working_dir('data_science')
init_sys_path()
apply_base_style()

# Configuration

In [ ]:
# Define relative paths
INPUT_FILE_ARTICLES = "../data_processed/articles.json"
INPUT_FILE_BLOG_POSTS = "../data_processed/blog_posts.json"
INPUT_FILE_EVENTS = "../data_processed/events.json"

# Simple check to verify the files are where we think they are
if os.path.exists(INPUT_FILE_ARTICLES):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_ARTICLES}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_ARTICLES}")

if os.path.exists(INPUT_FILE_BLOG_POSTS):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_BLOG_POSTS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_BLOG_POSTS}")

if os.path.exists(INPUT_FILE_EVENTS):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_EVENTS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_EVENTS}")

# Load raw data
These are already processed data, ready for analyses. They are stored in JSON format within the `data_processed/` folder (*or dummy examples* inside `data_processed_dummy/`).

### Load Articles Data from a JSON file & the main "Source of Truth" for Articles Data Analysis
This content is similar to blog posts; the primary difference is the ***storage format***.  
- ***Blog posts*** are a classical blog post data, while 
- ***articles*** are text created within Wix CMS table, and has no metrics (detecting *likes* and *views* for these cases require a workaround, which is time consuming).

In [ ]:
try:
    # Load the data
    _articles_raw = pd.read_json(INPUT_FILE_ARTICLES)
    print(f"✅ Loaded {len(_articles_raw)} Article records")

    articles_df = _articles_raw.copy()
    
    # 2. Output results
    print("\n📑 Available columns:", _articles_raw.columns.tolist())
    print("\n🎯 The main 'Source of Truth' for Articles Data Analysis")
    display(articles_df.head(2))

except Exception as e:
    print(f"❌ Error loading articles: {e}")

### Load Blog Posts Data from a JSON file

These are classical blog data, containing `metrics` (`likes`, `views`). 

In [ ]:
try:
    # 1. Load the data
    _blog_posts_raw = pd.read_json(INPUT_FILE_BLOG_POSTS)
    print(f"✅ Loaded {len(_blog_posts_raw)} Blog Post records.")
    
    # 2. Output results
    display(_blog_posts_raw.head(2))

except Exception as e:
    print(f"❌ Error loading blog posts: {e}")

#### Flattening `metrics`

In [ ]:
# 1. Create a fresh copy to work with
blog_posts_df = _blog_posts_raw.copy()

# 2. Flatten the metrics
# .fillna({}) ensures that if 'metrics' is None/NaN, it becomes an empty dict instead of crashing
metrics_flat = pd.json_normalize(blog_posts_df['metrics'].fillna({}))

# 3. Join back to the main dataframe
blog_posts_df = pd.concat([
    blog_posts_df.drop(columns=['metrics']).reset_index(drop=True),
    metrics_flat.reset_index(drop=True)
], axis=1)

# 4. Handle missing values in the new columns
# This turns NaN into 0 for the metrics columns
metric_cols = metrics_flat.columns
blog_posts_df[metric_cols] = blog_posts_df[metric_cols].fillna(0)

# 5. Output results
print(f"📑 Available columns: {blog_posts_df.columns.tolist()}")
display(blog_posts_df.head(2))

#### The main "Source of Truth" for Blog Posts Data Analysis & Primary `category`
[NOTE]   
There are blog posts with more than one categories. In these cases the *Primary Category* is selected with the `determine_primary_category` fucntion.  

In [ ]:
# 1. Primary Category 
blog_posts_df['Primary_Cat'] = blog_posts_df['categories'].apply(determine_primary_category)

# 2. Category Count (The "Tag Density")
# Counts how many categories were assigned to each post.
blog_posts_df['Cat_Count'] = blog_posts_df['categories'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# 3. Output results
print(f"📑 Available columns: {blog_posts_df.columns.tolist()}")

multi_tagged_posts = blog_posts_df[blog_posts_df['Cat_Count'] > 1]
print(f"\n🔍 Found {len(multi_tagged_posts)} posts with more than one category.")

print("\n🎯 The main 'Source of Truth' for Blog Posts Data Analysis")
display(blog_posts_df.head(2))

### Load Events Data from a JSON file

In [ ]:
try:
    # 1. Load the data
    _events_raw = pd.read_json(INPUT_FILE_EVENTS)
    print(f"✅ Loaded {len(_events_raw)} Event records.")

    # 2. Output results
    display(_events_raw.head(2))

except Exception as e:
    print(f"❌ Error loading events: {e}")

#### The main "Source of Truth" for Events Data Analysis & Flattening `eventGuests`

In [ ]:
# 1. Create a fresh copy to work with
events_df = _events_raw.copy()

# 2. Flatten the eventGuests
# .fillna({}) ensures that if 'eventGuests' is None/NaN, it becomes an empty dict instead of crashing
event_guests_flat = pd.json_normalize(events_df['eventGuests'].fillna({})).add_prefix('guest_')

# 3. Join back to the main dataframe
events_df = pd.concat([
    events_df.drop(columns=['eventGuests']).reset_index(drop=True),
    event_guests_flat.reset_index(drop=True)
], axis=1)

# 4. Handle missing values in the new columns
# This turns NaN into 0 for the eventGuests columns
metric_cols = event_guests_flat.columns
events_df[metric_cols] = events_df[metric_cols].fillna(0)

# 5. Output results
print(f"📑 Available columns: {events_df.columns.tolist()}")
print("\n🎯 The main 'Source of Truth' for Events Data Analysis")
display(events_df.head(2))  

# Convert date string to actual datetime objects

[NOTE]   
While `pd.read_json()` often auto-detects date strings, these columns are explicitly converted to ensure they are typed as `datetime64[ns]`.   
This guarantees consistency across different data sources and provides the necessary foundation for the temporal trend analysis.

In [ ]:
def convert_and_verify(df, col_name, label):
    print(f"--- {label} ---")

    # Check if it's already a datetime object
    if pd.api.types.is_datetime64_any_dtype(df[col_name]):
           print("💡 Note: Pandas already auto-converted this during loading!")

    print(f"Current format in memory: {df[col_name].iloc[0]}")
    print(f"Dtype: {df[col_name].dtype}\n")

    # We still run this to be safe (it won't hurt if already converted)
    df[col_name] = pd.to_datetime(df[col_name], errors='coerce')

convert_and_verify(articles_df, 'publishedDate', 'Articles')
convert_and_verify(blog_posts_df, 'publishedDate', 'Blog Posts')
convert_and_verify(events_df, 'date', 'Events')

# Plotting Table - Combined Data
[NOTE]  
The incomplete current year (2026) is excluded from the Combined DataFrame for global consistency.

In [ ]:
# 1. Prepare Articles 
articles_temp_df = articles_df[['publishedDate', 'title', 'category']].copy()
articles_temp_df['Type'] = 'Article'
articles_temp_df = articles_temp_df.rename(columns={'publishedDate': 'Date', 'category': 'Category',
'title': 'Title'})

# Fill metrics with 0 (since they aren't available)
for col in ['Blog_Views', 'Blog_Likes', 'Guest_Total', 'Guest_Going', 'Guest_Waitlist']:
    articles_temp_df[col] = 0

# 2. Prepare Blog Posts 
blog_temp_df = blog_posts_df[['publishedDate', 'title', 'Primary_Cat', 'views', 'likes']].copy()
blog_temp_df['Type'] = 'Blog Post'
blog_temp_df = blog_temp_df.rename(columns={'publishedDate': 'Date', 'Primary_Cat': 'Category', 'title': 'Title', 'views':
'Blog_Views', 'likes': 'Blog_Likes'})

# Fill event-specific metrics with 0
for col in ['Guest_Total', 'Guest_Going', 'Guest_Waitlist']:
    blog_temp_df[col] = 0

# 3. Prepare Events
events_temp_df = events_df[['date', 'title', 'category', 'guest_total', 'guest_going',
'guest_waitlist']].copy()
events_temp_df['Type'] = 'Event'
events_temp_df = events_temp_df.rename(columns={
    'date': 'Date',
    'category': 'Category',
    'title': 'Title',
    'guest_total': 'Guest_Total',
    'guest_going': 'Guest_Going',
    'guest_waitlist': 'Guest_Waitlist'
})

#  Fill blog-specific metrics with 0
for col in ['Blog_Views', 'Blog_Likes']:
    events_temp_df[col] = 0

# 4. Combine everything
combined_df = pd.concat([articles_temp_df, blog_temp_df, events_temp_df], ignore_index=True)

# 5. Feature Engineering
combined_df['Year'] = combined_df['Date'].dt.year
combined_df['Month'] = combined_df['Date'].dt.month

# Exclude incomplete current year (2026) for global consistency
combined_df = combined_df[combined_df['Year'] != 2026]

# 6. Final Check
print(f"📑 Available columns: {combined_df.columns.tolist()}")
print("\n🎯 The main 'Source of Truth' for Combined Data Analysis")
display(combined_df.sample(2))